## Manual Review of LLM Scope/Pillar Classifications (S5)

Turns `scope_LLM` / `confidence_LLM` / `pillar_LLM` (from `S3_LLM_scope.py`) into a trusted `scope_curated` / `pillar_curated` verdict on `funding_classified`. LLM output alone isn't ground truth - confident decisions are auto-inherited, everything else goes to a human.

Standalone notebook, not invoked by `pipeline_funding.py` - run it whenever there's new LLM-scored data to review, independent of any single pipeline run.

Auto-accept logic (no ML stage in Funding, so this is a plain confidence-band split, unlike Publications' confidence+ML-pillar-agreement rule):
- Confidence 5-7 with a real pillar assigned -> auto-approve (`scope_curated='in'`)
- Confidence 1 with no pillar assigned -> auto-reject (`scope_curated='out'`) - "reject" means excluded from being promoted onward, **not** deleted from `funding_classified` (deleting it would break S1's re-query dedup check)
- Confidence 2-4 -> manual review
- Anything with a missing/inconsistent scope, pillar, or LLM status also falls through to manual review, regardless of confidence

In [15]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

DB_PATH = Path('funding.db')  # self-contained in Pipeline/Funding, mirrors Publications
DATA_DIR = Path('data_review')
DATA_DIR.mkdir(exist_ok=True)

today = datetime.today().strftime("%y%m%d")
RUN_TIMESTAMP = datetime.today().strftime("%y%m%d_%H%M")  # for filenames, so reruns don't overwrite

## Pull LLM-scored grants for manual review

In [2]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [3]:
# Show tables in database
db.sql("SHOW TABLES")

┌────────────────────────────┐
│            name            │
│          varchar           │
├────────────────────────────┤
│ dim_query_test             │
│ dim_query_test_dedup       │
│ dimensions_2025_test       │
│ dimensions_2025_test_dedup │
│ funding_classified         │
│ funding_curated            │
└────────────────────────────┘

#### Get data from `funding_classified`

In [4]:
data = db.sql("SELECT * FROM funding_classified").df()

In [5]:
# Only rows that have actually been through S3 (LLM scoring)
data = data[data['status_LLM'].notna()]

In [6]:
# funding_classified grows forever, so re-running this notebook must not re-export rows
# already curated in an earlier pass - only look at rows with no scope_curated yet.
if 'scope_curated' in data.columns:
    data = data[data['scope_curated'].isna()]

print(f"{len(data)} LLM-scored grants awaiting curation")

32 LLM-scored grants awaiting curation


#### Create filter mask for curated scope and pillar

In [7]:
# decision mask for auto-accept vs manual review
auto_in_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'in') &
    (data['confidence_LLM'] >= 5) &
    (data['pillar_LLM'].notna()) & (data['pillar_LLM'] != 'NA')
)

auto_out_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'out') &
    (data['confidence_LLM'] == 1) &
    (data['pillar_LLM'] == 'NA')
)

inherit_mask = auto_in_mask | auto_out_mask

#### Assign curated scope/pillar or manual review

In [8]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated']  = np.where(inherit_mask, data['scope_LLM'],  'manual_review')
data['pillar_curated'] = np.where(inherit_mask, data['pillar_LLM'], 'manual_review')
data['date_review'] = today

In [9]:
# How many grants have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
in               13
out              11
manual_review     8
Name: count, dtype: int64

#### Save borderline grants for manual review

In [10]:
# Export grants for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(DATA_DIR / f'{RUN_TIMESTAMP}_funding_for_review.csv', index=False)
print(f"Saved {len(review)} grants for review -> {DATA_DIR / f'{RUN_TIMESTAMP}_funding_for_review.csv'}")

Saved 8 grants for review -> data\260722_1940_funding_for_review.csv


#### Save (automatically) curated scope and pillar back to `funding_classified`

In [11]:
# filter for the grants with a clear scope and pillar assignment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [12]:
# create columns in funding_classified if they don't already exist
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [13]:
# add to database
db.register('data', data[['Grant ID', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = data.scope_curated,
        pillar_curated = data.pillar_curated
    FROM data
    WHERE funding_classified."Grant ID" = data."Grant ID"
""")
print(f"Auto-curated {len(data)} grants in funding_classified")

Auto-curated 24 grants in funding_classified


In [14]:
db.close()

## Assign manually curated scope and pillar to respective grants

Run this section after you've opened the exported CSV, filled in real `scope_curated` / `pillar_curated` values (replacing `manual_review`) for each row, and saved it as `{timestamp}_funding_reviewed.csv` in the same `data/` folder.

In [16]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [20]:
# load manually reviewed data - update the date in this filename to match your reviewed export
reviewed_data = pd.read_csv(DATA_DIR / f'260722_1940_funding_reviewed.csv')  # <-- set to match the reviewed file you saved

#### Add scope and pillar back to `funding_classified`, using grant ID

In [21]:
# add to database
db.register('reviewed_data', reviewed_data[['Grant ID', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = reviewed_data.scope_curated,
        pillar_curated = reviewed_data.pillar_curated
    FROM reviewed_data
    WHERE funding_classified."Grant ID" = reviewed_data."Grant ID"
""")
print(f"Applied manual review decisions for {len(reviewed_data)} grants")

Applied manual review decisions for 8 grants


In [22]:
data = db.sql("SELECT * FROM funding_classified").df()
data[~data['scope_curated'].isna()]

,Grant ID,Title translated,Title,Abstract translated,Abstract,Researchers,Research Organization - standardized,Funding amount,Currency,Funding amount in USD,...,pillar_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status_LLM,stop_reason_LLM,date_LLM,scope_curated,pillar_curated
0,grant.15280337,Enrichment of zein from corn gluten and produc...,Anreichern von Zein aus Maiskleber und Herstel...,NaN,<NA>,NaN,Research Association of the German Food Industry,177877.0,EUR,177877.0,...,PB,True,False,False,False,ok,tool_use,260722,in,PB
1,grant.15218957,Evaluation of the Nursery Milk Scheme in England,Evaluation of the Nursery Milk Scheme in England,Our project will evaluate the nursery milk sch...,<NA>,"[{'first_name': 'Penny R', 'id': 'ur.010434454...",Sheffield Hallam University,825933.0,GBP,825933.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NA
2,grant.15060106,An Exclusive Sustainability Transition of Fash...,"""Een Exclusieve Duurzaamheidstransitie van de ...",This project takes the ecologically destructiv...,<NA>,"[{'first_name': 'Giselinde', 'id': 'ur.0146531...",KU Leuven,NaN,NaN,NaN,...,NA,False,False,False,False,ok,tool_use,260722,out,NA
3,grant.15149727,Masking Agents To Promote Ingestion Of Organic...,Masking Agents To Promote Ingestion Of Organic...,This STTR Phase I project is focused on novel ...,<NA>,NaN,Foresight Science & Technology (United States),181500.0,USD,181500.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NA
4,grant.15082917,Identification of Biomarkers of Plant-Rich Die...,Identification of Biomarkers of Plant-Rich Die...,Abstract: One potential mechanism through whic...,<NA>,"[{'first_name': 'Frank B', 'id': 'ur.013650315...",Harvard University,733785.0,USD,733785.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NA
5,grant.15145414,VegVit,VegVit,"""VegVit"" is an interdisciplinary project aimed...",<NA>,"[{'first_name': 'Myriam', 'id': 'ur.0164020000...",KU Leuven,NaN,NaN,NaN,...,PB,True,False,False,False,ok,tool_use,260722,in,PB
6,grant.15069166,Biologically Active Components of Human Milk,Biologically Active Components of Human Milk,PROJECT SUMMARY The 6th FASEB SRC conference o...,<NA>,"[{'first_name': 'Douglas Guy', 'id': 'ur.01127...",Federation of American Societies for Experimen...,10000.0,USD,10000.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NaN
7,grant.15201374,Novel Mechanism-of-Action (NMoA) genes: a path...,Novel Mechanism-of-Action (NMoA) genes: a path...,Implementing the latest knowledge of plant sci...,<NA>,"[{'first_name': 'Kamil', 'id': 'ur.0731530600....",University of East Anglia; 2Blades Foundation,954864.0,GBP,954864.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NA
8,grant.15150934,From Potato Waste to Mycoprotein – A Sustainab...,From Potato Waste to Mycoprotein – A Sustainab...,This project aims to upcycle food-grade potato...,<NA>,"[{'first_name': 'Carl', 'id': 'ur.013503127526...",Lund University,432546.0,SEK,432546.0,...,F,False,True,False,False,ok,tool_use,260722,in,F
9,grant.15150298,Microbial Feed from Swedish Biogas – Business ...,Microbial Feed from Swedish Biogas – Business ...,The project aims to operationalise the use of ...,<NA>,"[{'first_name': 'Henrik', 'id': 'ur.0100203046...",University of Gothenburg,408587.0,SEK,408587.0,...,NA,False,False,False,False,ok,tool_use,260722,out,NA


In [23]:
db.close()